# Final Model — Visual-only (Clusters + Faces)

This notebook builds the **final model only**.

**Pipeline:**
1. Load metadata + cluster predictions (from Torres replication capsule).
2. Resolve local image paths (Images2 folder).
3. Run **MTCNN** face detection to extract interpretable image features:
   - `face_count`, `face_count_hi`, `max_face_prob`
4. Train a **single final classifier** (logistic regression) on:
   - clusters (`predicted_labels`) + face features
5. Evaluate on a fixed 70/30 stratified split and save figures to `figures/`.

6. **Task**: predict `iscrowd` (crowd presence) from visual representation features. This is **not** a protest classifier.

**Reproducibility note:** Outputs are intentionally cleared so results from the earlier metadata model cannot be mistaken for this visual-only specification. Run all cells from the top using the pinned dependencies in `requirements.txt`; the authoritative results are reported in the README and final report.

In [ ]:
%pip install -q -r https://raw.githubusercontent.com/TemurAkhtamjonov/STATS201_project/main/requirements.txt

In [ ]:
#Setup: mount Drive and paths
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/STATS201_project/torres_replication/")
CAPSULE_DATA = DATA_DIR / "capsule" / "data"
IMAGES_DIR = CAPSULE_DATA / "Images2"
FIG_DIR = Path("/content/drive/MyDrive/STATS201_project/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_DIR exists:", DATA_DIR.exists())
print("CAPSULE_DATA exists:", CAPSULE_DATA.exists())
print("IMAGES_DIR exists:", IMAGES_DIR.exists())
print("FIG_DIR:", FIG_DIR)

In [ ]:
#Imports
import os, time, math
import numpy as np
import pandas as pd

from PIL import Image
import matplotlib.pyplot as plt
import cv2

import torch

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    ConfusionMatrixDisplay, f1_score
)

from facenet_pytorch import MTCNN
# Reproducibility (note: full determinism on GPU is not guaranteed)
torch.manual_seed(0)
np.random.seed(0)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

In [ ]:
# Initialize MTCNN.
# keep_all=True => return all faces (not just the largest one)
mtcnn = MTCNN(image_size=160, margin=20, keep_all=True, device=device)
print("MTCNN ready.")

In [ ]:
# Load and merge tabular data
meta_path = CAPSULE_DATA / "metadata_caravan_media_short.csv"
clusters_path = CAPSULE_DATA / "clusters_preds_caravan_newsapi.csv"

meta = pd.read_csv(meta_path, encoding="latin1")
clusters = pd.read_csv(clusters_path)

# clusters file has: file, predicted_labels
# meta file has: imageid, iscrowd

clusters["imageid"] = clusters["file"].astype(str).str.replace(r"\.p$", "", regex=True)

df = meta.merge(clusters[["imageid", "predicted_labels"]], on="imageid", how="inner")
df["y"] = df["iscrowd"].astype(int)

print("Merged df shape:", df.shape)
print("y distribution:", df["y"].value_counts(normalize=True).round(3).to_dict())
df.head(3)

In [ ]:
# Resolve image paths
# In this dataset, `imageid` already matches the filename inside Images2.
def find_image_path(imageid: str):
    p = IMAGES_DIR / str(imageid)
    return str(p) if p.exists() else None

df["image_path"] = df["imageid"].astype(str).apply(find_image_path)

n_found = df["image_path"].notna().sum()
print(f"Images found: {n_found} / {len(df)}")
if n_found < len(df):
    display(df[df["image_path"].isna()][["imageid"]].head(10))

In [ ]:
#Helper: show image (optional)
def show_img(img, title=None, figsize=(6,6)):
    plt.figure(figsize=figsize)
    if isinstance(img, Image.Image):
        plt.imshow(img)
    else:
        plt.imshow(img)
    plt.axis("off")
    if title:
        plt.title(title)
    plt.show()

In [ ]:
#Face feature extraction (MTCNN)
# We compute:
# - face_count: number of detected faces (any confidence)
# - face_count_hi: number of faces with prob >= HI_THR
# - max_face_prob: max probability among detected faces (0 if none)

HI_THR = 0.90

def extract_face_features(img_path: str, hi_thr: float = HI_THR):
    # Safety: img_path must be a string path
    if not isinstance(img_path, str):
        return {"face_count": 0, "face_count_hi": 0, "max_face_prob": 0.0}

    try:
        img = Image.open(img_path).convert("RGB")
    except Exception:
        # corrupted / unreadable image
        return {"face_count": 0, "face_count_hi": 0, "max_face_prob": 0.0}

    boxes, probs = mtcnn.detect(img)  # probs: array of confidence scores
    if probs is None:
        return {"face_count": 0, "face_count_hi": 0, "max_face_prob": 0.0}

    probs = np.array(probs, dtype=float)
    face_count = int(len(probs))
    face_count_hi = int((probs >= hi_thr).sum())
    max_face_prob = float(probs.max()) if face_count > 0 else 0.0

    return {
        "face_count": face_count,
        "face_count_hi": face_count_hi,
        "max_face_prob": max_face_prob,
    }

In [ ]:
# Run extraction over all images (this can take several minutes on CPU)
t0 = time.time()

face_rows = []
paths = df["image_path"].tolist()

for i, p in enumerate(paths, start=1):
    face_rows.append(extract_face_features(p))
    if i % 50 == 0:
        print(f"done {i}/{len(paths)}")

face_feats = pd.DataFrame(face_rows, index=df.index)
df[["face_count", "face_count_hi", "max_face_prob"]] = face_feats

df["max_face_prob"] = df["max_face_prob"].fillna(0.0)

print("time (sec):", round(time.time() - t0, 2))
df[["imageid", "y", "face_count", "face_count_hi", "max_face_prob"]].head()

In [ ]:
#Quick diagnostics figure: face count by label (log scale)
fc0 = df.loc[df["y"]==0, "face_count"].values
fc1 = df.loc[df["y"]==1, "face_count"].values

plt.figure(figsize=(7,4))
plt.boxplot([fc0, fc1], tick_labels=["iscrowd=0","iscrowd=1"], showfliers=True)
plt.yscale("log")  # extreme outliers exist
plt.ylabel("Detected face_count (log scale)")
plt.title("Face count distribution by crowd label")
plt.tight_layout()
out_path = FIG_DIR / "facecount_by_label_boxplot.png"
plt.savefig(out_path, dpi=200)
plt.show()
print("Saved:", out_path)

In [ ]:
#Final model (single model): clusters + face features ===
# Feature set:
# - predicted_labels (cluster id)
# - face_count, face_count_hi, max_face_prob (numeric)

X = df[["predicted_labels", "face_count", "face_count_hi", "max_face_prob"]].copy()
y = df["y"].astype(int)

# Impute numeric face features
X["face_count"] = X["face_count"].fillna(0)
X["face_count_hi"] = X["face_count_hi"].fillna(0)
X["max_face_prob"] = X["max_face_prob"].fillna(0.0)

cat_cols = ["predicted_labels"]
num_cols = ["face_count", "face_count_hi", "max_face_prob"]

pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", "passthrough", num_cols),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

final_model = Pipeline([
    ("pre", pre),
    ("logit", LogisticRegression(max_iter=5000, class_weight="balanced"))
])

final_model.fit(X_train, y_train)
pred = final_model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, pred), 4))
print(classification_report(y_test, pred, digits=3))

In [ ]:
#Baseline model: clusters only (ablation)
Xb = df[["predicted_labels"]].copy()
yb = df["y"].astype(int)

pre_b = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["predicted_labels"]),
])

Xb_train, Xb_test, yb_train, yb_test = train_test_split(
    Xb, yb, test_size=0.30, random_state=42, stratify=yb
)

baseline = Pipeline([
    ("pre", pre_b),
    ("logit", LogisticRegression(max_iter=5000, class_weight="balanced"))
])

baseline.fit(Xb_train, yb_train)
pred_b = baseline.predict(Xb_test)

print("Baseline Accuracy:", round(accuracy_score(yb_test, pred_b), 4))
print("Baseline F1:", round(f1_score(yb_test, pred_b), 4))

In [ ]:
#Confusion matrix figure
cm = confusion_matrix(y_test, pred, labels=[0,1])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0,1])

fig, ax = plt.subplots(figsize=(5,4))
disp.plot(ax=ax, values_format="d")
ax.set_title("Confusion Matrix (Test) – clusters + face features")
plt.tight_layout()
out_path = FIG_DIR / "confusion_matrix_clusters_faces_model.png"
plt.savefig(out_path, dpi=200)
plt.show()
print("Saved:", out_path)
print("CM:\n", cm)

In [ ]:
#Save a compact results row (for reporting)
def summarize_row(name, y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0,1])
    tn, fp, fn, tp = cm.ravel()
    # Precision/recall/F1 for class 1 (crowd)
    from sklearn.metrics import precision_recall_fscore_support
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[0,1], zero_division=0
    )
    return {
        "model": name,
        "n_test": int(len(y_true)),
        "acc": float(accuracy_score(y_true, y_pred)),
        "prec_1": float(p[1]),
        "rec_1": float(r[1]),
        "f1_1": float(f1[1]),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }

results = pd.DataFrame([
    summarize_row("Baseline (clusters only)", yb_test, pred_b),
    summarize_row("Final (clusters + faces)", y_test, pred),
]).round(4)
results

In [ ]:
# Optional: export the enriched dataframe (with face features) for reuse
cache_csv = DATA_DIR / "df_with_face_features.csv"   # put in data dir, not figures
if cache_csv.exists():
    df = pd.read_csv(cache_csv)
else:
    # run extraction ...
    df.to_csv(cache_csv, index=False)